# R3maJ on Colab (v2)

1v1 Rocket League self-play trainer (GigaLearnCPP + RLGymCPP), **Nexto-exact rewards**, GPU training.

**How to run:**
1. Ensure **Runtime -> Change runtime type -> T4 GPU** is selected, then restart.
2. Run cells top to bottom.
3. Cell 2 mounts your Drive (click the auth link once). Checkpoints auto-backup to `My Drive/R3maJ/checkpoints/` from the start.
4. Optional: upload the converted replay binary to `My Drive/R3maJ/serialized_replays.bin` and cell 2 copies it into the build dir - training then starts ~70% of resets from human replay states (height-weighted sampling).
5. Use the last cell (STOP) to stop training cleanly - it saves a checkpoint first.

Source: `https://github.com/vfxjamer/R3maJ.git`

In [ ]:
# 1. Clone source (self-contained: src + CMakeLists + collision_meshes + GigaLearnCPP thirdparty)
import os, subprocess, shutil

ROOT = "/content/R3maJ"
REPO = "https://github.com/vfxjamer/R3maJ.git"

if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "pull"], check=False)

print("ROOT:", ROOT)
print("contents:", sorted(os.listdir(ROOT)))

In [ ]:
# 2. Mount Google Drive EARLY (so checkpoints back up from the start) + restore latest checkpoint.
import os, shutil, glob

if not os.path.isdir("/content/drive"):
    print("mounting Drive (complete the auth popup in the browser)...", flush=True)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("WARN: drive mount failed:", e, "- backups will retry automatically", flush=True)
else:
    print("drive already mounted", flush=True)

DRIVE_CKPT = "/content/drive/MyDrive/R3maJ/checkpoints"
LOCAL_CKPT = "/content/R3maJ/build/checkpoints"
DRIVE_REPLAY = "/content/drive/MyDrive/R3maJ/serialized_replays.bin"
LOCAL_REPLAY = "/content/R3maJ/build/serialized_replays.bin"

os.makedirs(LOCAL_CKPT, exist_ok=True)

def ts_of(d):
    try: return int(os.path.basename(d))
    except Exception: return -1

if os.path.isdir(DRIVE_CKPT):
    drive_dirs = sorted([d for d in glob.glob(os.path.join(DRIVE_CKPT, "*")) if os.path.isdir(d)], key=ts_of)
    print(f"Drive has {len(drive_dirs)} checkpoint dirs")
    if drive_dirs:
        newest = drive_dirs[-1]
        dest = os.path.join(LOCAL_CKPT, os.path.basename(newest))
        if not os.path.isdir(dest):
            print("restoring latest checkpoint:", os.path.basename(newest))
            shutil.copytree(newest, dest)
else:
    print("no Drive checkpoints yet")

if os.path.exists(DRIVE_REPLAY) and not os.path.exists(LOCAL_REPLAY):
    print("copying replay binary from Drive")
    shutil.copy(DRIVE_REPLAY, LOCAL_REPLAY)

print("local checkpoints:", sorted(os.listdir(LOCAL_CKPT)) if os.path.isdir(LOCAL_CKPT) else [])
print("replay binary present:", os.path.exists(LOCAL_REPLAY))

In [ ]:
# 3a. ALIGN source to the reward curriculum (idempotent, mostly a sanity check now).
# The phase-based reward curriculum now lives IN THE REPO (src/PhaseManager.cpp +
# src/main.cpp), so Colab just clones it:
#   0-5B   -> Phase 0 (contact/fundamentals)
#   5B-15B -> Phase 1 (ball-to-goal)
#   15B-30B-> Phase 2 (basic 1v1)
#   30B+   -> Phase 3 (mechanical; rewards ramp 0 -> Necto targets between 30B and 60B)
# Phase is auto-selected from the loaded checkpoint's total_timesteps, so no CLI --phase
# or reward-weight patching is needed on Colab anymore.
import os
ROOT = "/content/R3maJ"

def _has(path, needle, nice):
    if not os.path.exists(path):
        print("MISS:", path); return
    s = open(path).read()
    ok = needle in s
    print(("OK  " if ok else "MISSING ") + nice, "<-", os.path.relpath(path, ROOT))

pm = os.path.join(ROOT, "src", "PhaseManager.cpp")
_has(pm, "MECH_RAMP_END",       "mech ramp window")
_has(pm, "PhaseManager::PhaseManager()", "phase table")

main = os.path.join(ROOT, "src", "main.cpp")
_has(main, "GetRewards(g_totalTimesteps)", "auto phase selection")

kpd = os.path.join(ROOT, "thirdparty", "GigaLearnCPP-Leak", "GigaLearnCPP", "src", "public", "GigaLearnCPP", "Util", "KeyPressDetector.cpp")
_has(kpd, "R3maJ: keypress detector disabled", "keypress disabled")

print("Curriculum markers checked - the repo clone is already curriculum-ready.")

In [ ]:
# 2b. wandb setup: install the Python package the embedded interpreter will use, and set the API key.
import os, subprocess
WANDB_API_KEY = "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"

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_MODE"] = "online"
print("installing wandb...")
r = subprocess.run(["pip", "install", "-q", "wandb"], capture_output=True, text=True)
print("pip rc:", r.returncode, (r.stderr or "")[-200:])
import wandb
print("wandb:", wandb.__version__)
print("key set:", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
# 2. Install build deps (apt) + ensure GPU-enabled torch via pip if needed
import subprocess, sys, os

APT_PKGS = ["build-essential", "cmake", "git", "libpython3-dev", "pkg-config"]
print("apt update/install...")
r = subprocess.run(["apt-get", "update", "-qq"] + ["&&"] if False else ["apt-get", "update", "-qq"], capture_output=True, text=True)
print("update rc:", r.returncode, (r.stderr or "")[-300:])
r = subprocess.run(["apt-get", "install", "-y", "-qq"] + APT_PKGS, capture_output=True, text=True)
print("install rc:", r.returncode, (r.stderr or "")[-300:])

# torch: pip's libtorch headers are used by CMake (TORCH_INSTALL_PREFIX).
# On a GPU runtime ensure CUDA-enabled torch. On CPU runtime keep whatever is there.
import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available(), "cuda build:", torch.version.cuda)
if torch.version.cuda is None:
    print("NOTE: torch is CPU-only. Training cell will require switching to GPU runtime.")

cmake_v = subprocess.run(["cmake", "--version"], capture_output=True, text=True).stdout
print(cmake_v.splitlines()[0])
print("gcc:", subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
print("python dev headers:", os.path.exists("/usr/include/python3.12/Python.h") or os.path.exists("/usr/local/include/python3.12/Python.h") or os.path.exists("/usr/include/python3.11/Python.h"))

In [ ]:
# 3. Configure + build R3maJ (Release). Override TORCH_INSTALL_PREFIX to pip's torch location.
import os, subprocess, sys

ROOT = "/content/R3maJ"
os.chdir(ROOT)

import torch
torch_prefix = os.path.dirname(torch.__file__)  # contains share/cmake/Torch
print("TORCH_INSTALL_PREFIX =", torch_prefix)
print("has Torch cmake:", os.path.exists(os.path.join(torch_prefix, "share", "cmake", "Torch")))

configure = [
    "cmake", "-S", ".", "-B", "build",
    "-DCMAKE_BUILD_TYPE=Release",
    f"-DTORCH_INSTALL_PREFIX={torch_prefix}",
]
print(" ".join(configure))
r = subprocess.run(configure, capture_output=True, text=True)
print("configure rc:", r.returncode)
print((r.stdout or "")[-2500:])
print((r.stderr or "")[-1500:])

In [ ]:
# 4. Build (this is the long step). Uses all available cores.
import os, subprocess

ROOT = "/content/R3maJ"
os.chdir(ROOT)

nproc = os.cpu_count() or 2
print(f"Building with -j{nproc} ... (this can take 10-30 min)")
r = subprocess.run(["cmake", "--build", "build", "-j", str(nproc)], capture_output=True, text=True)
print("build rc:", r.returncode)
tail = (r.stdout or "")[-3000:] + (r.stderr or "")[-2000:]
print(tail)
print("---")
exe = os.path.join(ROOT, "build", "R3maJ")
print("binary exists:", os.path.exists(exe), exe if os.path.exists(exe) else "")

In [ ]:
import sys, platform
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch import:", e)

In [ ]:
# 4b. System performance monitor -> wandb (GPU util/VRAM/temp, CPU, RAM, disk I/O)
# Runs in the background while training. Uses the SAME wandb run via run_id from the binary log.
import os, time, threading, subprocess, glob, re

os.environ["WANDB_API_KEY"] = WANDB_API_KEY

def _read_run_id():
    logs = glob.glob("/content/R3maJ/build/*.log") + glob.glob("/content/R3maJ/*.log")
    for f in logs:
        try:
            txt = open(f, errors="ignore").read()
            m = re.search(r'run with ID : "([^"]+)"', txt)
            if m: return m.group(1)
        except Exception:
            pass
    return None

def _nvidia():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout.strip()
        vals = out.split(",")
        return dict(gpu_util=float(vals[0]), vram_used_mb=float(vals[1]), vram_total_mb=float(vals[2]), gpu_temp_c=float(vals[3]))
    except Exception:
        return None

def _cpu():
    try:
        import psutil
        return dict(cpu_util=psutil.cpu_percent(interval=1), ram_used_gb=round(psutil.virtual_memory().used / 1e9, 2),
                    ram_total_gb=round(psutil.virtual_memory().total / 1e9, 2))
    except Exception:
        return None

def sys_monitor(project, group):
    import wandb
    while not os.path.exists("/content/R3maJ/build/R3maJ"):
        time.sleep(10)
    run_id = None
    for _ in range(600):  # wait up to 1h for the training binary to start
        run_id = _read_run_id()
        if run_id: break
        time.sleep(6)
    if not run_id:
        print("system monitor: training never started (no run id found)")
        return
    run = wandb.init(project=project, group=group, name="system-monitor", id=run_id, resume=True)
    print(f"system monitor attached to run {run_id}")
    while True:
        d = {}
        g, c = _nvidia(), _cpu()
        if g: d.update({("sys/" + k): v for k, v in g.items()})
        if c: d.update({("sys/" + k): v for k, v in c.items()})
        if d: run.log(d)
        time.sleep(10)

t = threading.Thread(target=sys_monitor, args=("r3maj", "phases"), daemon=True)
t.start()
print("system monitor thread started")

In [ ]:
# TRAINING SUPERVISOR
#   --phase -1 (default): auto-select phase from the resume checkpoint's total_timesteps
# - Auto-selects device: CUDA if available, otherwise CPU (slow but runnable).
# - Starts the 24/7 backup daemon + a live log tailer, then launches training.
# - The tailer streams the binary's real-time output (steps/timesteps/reports) into THIS cell.
# - If the binary crashes, it auto-restarts with backoff, resuming from the latest checkpoint.
import os, sys, subprocess, torch, time, threading, shutil, glob

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("WARNING: no GPU detected - training on CPU (will be slow)", flush=True)

BUILD = "/content/R3maJ/build"
LOCAL_CKPT = os.path.join(BUILD, "checkpoints")
DRIVE_CKPT = "/content/drive/MyDrive/R3maJ/checkpoints"

# ---- 24/7 backup daemon (guarded: one instance per kernel) ----
def _ts(d):
    try: return int(os.path.basename(d))
    except Exception: return -1

def _backup_once():
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    local = sorted([d for d in glob.glob(os.path.join(LOCAL_CKPT, "*")) if os.path.isdir(d)], key=_ts)
    drive = set(os.path.basename(d) for d in glob.glob(os.path.join(DRIVE_CKPT, "*")))
    for d in local:
        name = os.path.basename(d)
        if name in drive:
            continue
        tmp = os.path.join(DRIVE_CKPT, name + ".tmp")
        try:
            print(f"[backup] uploading checkpoint {name} ...", flush=True)
            shutil.copytree(d, tmp)
            shutil.move(tmp, os.path.join(DRIVE_CKPT, name))
            print(f"[backup] {name} uploaded", flush=True)
        except Exception as e:
            print("[backup] err:", e, flush=True)
            shutil.rmtree(tmp, ignore_errors=True)

def _backup_loop():
    while True:
        try:
            if os.path.isdir("/content/drive"):
                _backup_once()
            else:
                print("[backup] drive not mounted, retrying in 60s", flush=True)
        except Exception as e:
            print("[backup] loop err:", e, flush=True)
        time.sleep(60)

if "_R3MAJ_BACKUP_DAEMON_" not in globals():
    globals()["_R3MAJ_BACKUP_DAEMON_"] = True
    threading.Thread(target=_backup_loop, daemon=True).start()
    print("[backup] 24/7 daemon started (new checkpoints -> Drive within 60s)", flush=True)

# ---- Live log tailer: streams train.log (binary output + history) into this cell ----
_tailer_lock = {"run": True}
if "_R3MAJ_TAILER_" not in globals():
    globals()["_R3MAJ_TAILER_"] = True
    def _tail_loop():
        path = os.path.join(BUILD, "train.log")
        while not os.path.exists(path):
            time.sleep(1)
        while True:
            try:
                with open(path, "r", errors="ignore") as f:
                    while True:
                        data = f.read()
                        if data:
                            sys.stdout.write(data)
                            sys.stdout.flush()
                        else:
                            if not _tailer_lock.get("run", True):
                                return
                            time.sleep(0.5)
            except SystemExit:
                return
            except Exception:
                time.sleep(1)
    threading.Thread(target=_tail_loop, daemon=True).start()
    print("[tailer] streaming binary output to this cell...", flush=True)

# ---- Training supervisor ----
os.chdir(BUILD)
os.makedirs(LOCAL_CKPT, exist_ok=True)

REPLAY_ARG = ["--replays", "serialized_replays.bin"] if os.path.exists("serialized_replays.bin") else []
BASE_CMD = ["stdbuf", "-oL", "-eL", "./R3maJ", "--device", DEVICE, "--phase", "-1", "--save-dir", "checkpoints", "--games", "192"] + REPLAY_ARG
print("BASE CMD:", " ".join(BASE_CMD), flush=True)

backoff = 10
attempt = 0
with open("train.log", "a") as log:
    while True:
        attempt += 1
        t0 = time.time()
        print(f"[supervisor] launch #{attempt}: {' '.join(BASE_CMD[:6])} ...", flush=True)
        proc = subprocess.Popen(BASE_CMD, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        rc = proc.wait()
        elapsed = time.time() - t0
        if rc == 0:
            print("[supervisor] clean exit (rc=0) â€” stopping supervisor.", flush=True)
            break
        print(f"[supervisor] crashed rc={rc} after {elapsed:.0f}s â€” restarting in {backoff}s", flush=True)
        time.sleep(backoff)
        if elapsed < 60:
            backoff = min(backoff * 2, 600)
        else:
            backoff = 10
print("supervisor exited.")

In [ ]:
# STATUS: drive mounted? binary running? checkpoints present?
import os, subprocess, glob

print("drive mounted:", os.path.isdir("/content/drive"))
if os.path.isdir("/content/drive"):
    ck = "/content/drive/MyDrive/R3maJ/checkpoints"
    print("drive ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(ck + "/*")) if os.path.isdir(ck) else "none yet")

LOCAL = "/content/R3maJ/build/checkpoints"
print("local ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(LOCAL + "/*")) if os.path.isdir(LOCAL) else "none yet")

r = subprocess.run(["pgrep", "-af", "R3maJ"], capture_output=True, text=True)
print("R3maJ processes:", r.stdout.strip() if r.stdout.strip() else "none")

In [ ]:
# STOP training cleanly: SIGTERM (binary saves a checkpoint then exits), SIGKILL fallback.
import subprocess, time
r = subprocess.run(["pkill", "-TERM", "-f", "R3maJ"], capture_output=True, text=True)
print("SIGTERM sent:", r.returncode == 0)
time.sleep(20)
r2 = subprocess.run(["pgrep", "-af", "R3maJ"], capture_output=True, text=True)
alive = r2.stdout.strip()
if alive:
    print("still alive, force killing:", alive)
    subprocess.run(["pkill", "-9", "-f", "R3maJ"], capture_output=True)
    time.sleep(3)
r3 = subprocess.run(["pgrep", "-af", "R3maJ"], capture_output=True, text=True)
print("remaining:", r3.stdout.strip() if r3.stdout.strip() else "none")